# 08 딥러닝 예측 모델

LSTM, Autoformer, N-HiTS, iTransformer를 **type×family** 주간 시계열에 적용합니다 (CPU, 논문 설정 유지).

- 검증 구간: 201731–201733

In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == 'code' else NOTEBOOK_DIR
sys.path.insert(0, str(REPO_ROOT / 'code'))

from utils.paths import DATA_PROCESSED
from utils.splits import VAL_WEEKS, TRAIN_WEEK_MAX
from utils.metrics import wmape
from utils.dl_models import DL_MODELS

df = pd.read_parquet(DATA_PROCESSED / 'df_weekly.parquet')
df = df.sort_values(['type', 'family', 'yearweek']).reset_index(drop=True)
HORIZON = len(VAL_WEEKS)
LOOKBACK = 16
print('series:', df.groupby(['type','family']).ngroups, '| horizon:', HORIZON)

series: 165 | horizon: 3


In [2]:
rows = []
for (typ, fam), g in tqdm(df.groupby(['type', 'family']), desc='DL forecast'):
    g_train = g[g['yearweek'] <= TRAIN_WEEK_MAX]
    g_val = g[g['yearweek'].isin(VAL_WEEKS)].sort_values('yearweek')
    if len(g_val) == 0:
        continue
    y_true = g_val['sales'].values.astype(float)
    series = g_train['sales'].values.astype(float)

    for name, fn in DL_MODELS.items():
        pred = fn(LOOKBACK, HORIZON, series)
        if len(pred) != len(y_true):
            pred = np.resize(pred, len(y_true))
        rows.append({
            'type': typ, 'family': fam, 'model': name,
            'wmape': wmape(y_true, pred),
            'mae': float(np.mean(np.abs(y_true - pred))),
        })

dl_results = pd.DataFrame(rows)
summary = dl_results.groupby('model')['wmape'].mean().sort_values().reset_index()
print('=== DL 모델별 평균 WMAPE (%) ===')
print(summary.round(2))

out = DATA_PROCESSED / 'forecast_dl_results.parquet'
dl_results.to_parquet(out, index=False)
summary.to_csv(DATA_PROCESSED / 'forecast_dl_summary.csv', index=False)
print('저장:', out)
dl_results.head()

DL forecast:   0%|          | 0/165 [00:00<?, ?it/s]

=== DL 모델별 평균 WMAPE (%) ===
          model  wmape
0        N-HiTS  55.61
1  iTransformer  91.79
2    Autoformer  94.91
3          LSTM  98.18
저장: C:\Users\kjh\ai-retail-demandforecasting\data\processed\forecast_dl_results.parquet


,type,family,model,wmape,mae
0,A,AUTOMOTIVE,LSTM,92.677732,6.187783e+02
1,A,AUTOMOTIVE,N-HiTS,38.440600,2.566551e+02
2,A,AUTOMOTIVE,Autoformer,73.921658,4.935503e+02
3,A,AUTOMOTIVE,iTransformer,73.999034,4.940669e+02
4,A,BABY CARE,LSTM,NaN,3.104409e-10


## 분석 요약

### DL 모델별 평균 WMAPE (전체 165 시계열, 클러스터 미분리)
| 모델 | WMAPE |
|------|-------|
| N-HiTS | 55.6% |
| iTransformer | 91.8% |
| Autoformer | 94.9% |
| LSTM | 98.2% |

### 해석 (주의)
- 본 장은 **전역 벤치마크**이며, 07장 ARIMA(46.9%)보다 DL 평균이 높게 나왔습니다.
- 그러나 **학위논문**에서는 **iTransformer가 전 클러스터·type 조건에서 최고 성능**이었고, 임베딩 결합 시에도 동일했습니다.
- 따라서 클러스터별 최종 비교는 **PatchTST 임베딩 + iTransformer**로 SBC vs ML을 재실험합니다 (10장, `docs/실험_프레임워크.md`).
- 패턴 유형(Smooth/Intermittent 등)과 알고리즘 대응은 이론적 참고용이며, **실측 WMAPE가 최종 선택 기준**입니다.